In [47]:
import json
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

In [49]:
# ========= Paths =========
INPUT_FILE = Path("../../raw_data/Basal v2.xlsx")

OUTPUT_DIR = Path("../../results/processed_hrv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_CSV = OUTPUT_DIR / "basal_v2_clean.csv"
CLEAN_XLSX = OUTPUT_DIR / "basal_v2_clean.xlsx"
PROCESSING_JSON = OUTPUT_DIR / "processing_log.json"

INITIAL_DIAGNOSTIC_CSV = OUTPUT_DIR / "initial_diagnostic.csv"
FINAL_DIAGNOSTIC_CSV = OUTPUT_DIR / "final_diagnostic.csv"
HEIGHT_INCONSISTENCIES_CSV = OUTPUT_DIR / "height_inconsistencies.csv"
BMI_INCONSISTENCIES_CSV = OUTPUT_DIR / "bmi_inconsistencies.csv"
HR_RR_INCONSISTENCIES_CSV = OUTPUT_DIR / "hr_rr_inconsistencies.csv"
RANGE_FLAGS_CSV = OUTPUT_DIR / "range_flags.csv"

In [50]:
# ========= Load raw data =========
df_raw = pd.read_excel(INPUT_FILE).copy()
df = df_raw.copy()

print("Raw shape:", df.shape)
print("\nRaw columns:")
print(df.columns.tolist())

Raw shape: (530, 20)

Raw columns:
['Unnamed: 0', 'Unnamed: 1', 'redcap_repeat_instrument', 'sex', 'age', 't2m_pre_mean_rr', 't2m_pre_mean_hr', 't2m_pre_sdnn', 't2m_pre_rmssd', 't2m_pre_hf', 't2m_pre_lf', 't2m_pre_vlf', 'bp_systolic', 'bp_diastolic', 'bp_pam', 'bp_pp', 'imc', 'weight kg', 'height cm', 'height mt']


In [51]:
# ========= Processing log =========
processing_log = {
    "dataset_name": "Basal v2",
    "timestamp": datetime.now().isoformat(),
    "input_file": str(INPUT_FILE),
    "initial_shape": {
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
    },
    "steps": []
}

def add_step(step_name, description, details=None):
    processing_log["steps"].append({
        "step": step_name,
        "description": description,
        "details": details or {}
    })

In [52]:
# ========= Helper functions =========
def clean_numeric_string(x):
    """
    Clean malformed numeric strings such as:
    '15 .9', '  7.0 ', '1,75', etc.
    Returns np.nan for empty or invalid values.
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "":
        return np.nan

    # remove inner spaces
    x = re.sub(r"\s+", "", x)

    # replace comma decimal separator if present
    x = x.replace(",", ".")

    try:
        return float(x)
    except ValueError:
        return np.nan


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

In [53]:
# ========= Initial diagnostic =========
initial_diagnostic = pd.DataFrame({
    "dtype_raw": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True)
}).sort_values(["missing_n", "dtype_raw"], ascending=[False, True])

initial_diagnostic.to_csv(INITIAL_DIAGNOSTIC_CSV)
initial_diagnostic

,dtype_raw,missing_n,missing_pct,n_unique
Unnamed: 0,float64,530,100.00,0
Unnamed: 1,float64,530,100.00,0
redcap_repeat_instrument,float64,530,100.00,0
t2m_pre_sdnn,float64,11,2.08,268
t2m_pre_rmssd,object,8,1.51,278
t2m_pre_mean_hr,float64,1,0.19,55
t2m_pre_mean_rr,float64,0,0.00,303
bp_pam,float64,0,0.00,134
imc,float64,0,0.00,418
weight kg,float64,0,0.00,210


In [54]:
# ========= Drop fully empty columns =========
fully_empty_cols = [col for col in df.columns if df[col].isna().all()]

df = df.drop(columns=fully_empty_cols)

add_step(
    step_name="drop_fully_empty_columns",
    description="Removed columns containing only missing values.",
    details={
        "dropped_columns": fully_empty_cols,
        "n_dropped": len(fully_empty_cols)
    }
)

print("Dropped fully empty columns:", fully_empty_cols)
print("Shape after dropping empty columns:", df.shape)

Dropped fully empty columns: ['Unnamed: 0', 'Unnamed: 1', 'redcap_repeat_instrument']
Shape after dropping empty columns: (530, 17)


In [55]:
# ========= Standardize column names =========
rename_map = {
    "weight kg": "weight_kg",
    "height cm": "height_cm",
    "height mt": "height_m"
}

df = df.rename(columns=rename_map)

add_step(
    step_name="rename_columns",
    description="Renamed columns for consistency and easier downstream processing.",
    details={"rename_map": rename_map}
)

print(df.columns.tolist())

['sex', 'age', 't2m_pre_mean_rr', 't2m_pre_mean_hr', 't2m_pre_sdnn', 't2m_pre_rmssd', 't2m_pre_hf', 't2m_pre_lf', 't2m_pre_vlf', 'bp_systolic', 'bp_diastolic', 'bp_pam', 'bp_pp', 'imc', 'weight_kg', 'height_cm', 'height_m']


In [56]:
# ========= Clean malformed numeric-like columns =========
object_cols = df.select_dtypes(include="object").columns.tolist()

object_cols_before = object_cols.copy()

for col in object_cols:
    df[col] = df[col].apply(clean_numeric_string)

add_step(
    step_name="clean_object_numeric_strings",
    description="Applied generic malformed numeric-string cleaning to object columns.",
    details={
        "object_columns_cleaned": object_cols_before
    }
)

df.dtypes

sex                  int64
age                  int64
t2m_pre_mean_rr    float64
t2m_pre_mean_hr    float64
t2m_pre_sdnn       float64
t2m_pre_rmssd      float64
t2m_pre_hf           int64
t2m_pre_lf           int64
t2m_pre_vlf          int64
bp_systolic          int64
bp_diastolic         int64
bp_pam             float64
bp_pp                int64
imc                float64
weight_kg          float64
height_cm            int64
height_m           float64
dtype: object

In [57]:
# ========= Force numeric conversion =========
numeric_cols = [
    "sex",
    "age",
    "t2m_pre_mean_rr",
    "t2m_pre_mean_hr",
    "t2m_pre_sdnn",
    "t2m_pre_rmssd",
    "t2m_pre_hf",
    "t2m_pre_lf",
    "t2m_pre_vlf",
    "bp_systolic",
    "bp_diastolic",
    "bp_pam",
    "bp_pp",
    "imc",
    "weight_kg",
    "height_cm",
    "height_m",
]

existing_numeric_cols = [col for col in numeric_cols if col in df.columns]

for col in existing_numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

add_step(
    step_name="force_numeric_conversion",
    description="Forced selected variables to numeric type using coercion.",
    details={
        "numeric_columns": existing_numeric_cols
    }
)

df[existing_numeric_cols].dtypes

sex                  int64
age                  int64
t2m_pre_mean_rr    float64
t2m_pre_mean_hr    float64
t2m_pre_sdnn       float64
t2m_pre_rmssd      float64
t2m_pre_hf           int64
t2m_pre_lf           int64
t2m_pre_vlf          int64
bp_systolic          int64
bp_diastolic         int64
bp_pam             float64
bp_pp                int64
imc                float64
weight_kg          float64
height_cm            int64
height_m           float64
dtype: object

In [58]:
# ========= Missingness in key variables =========
key_missing = pd.DataFrame({
    "missing_n": df[existing_numeric_cols].isna().sum(),
    "missing_pct": (df[existing_numeric_cols].isna().mean() * 100).round(2)
}).sort_values("missing_n", ascending=False)

key_missing

,missing_n,missing_pct
t2m_pre_sdnn,11,2.08
t2m_pre_rmssd,8,1.51
t2m_pre_mean_hr,1,0.19
sex,0,0.00
t2m_pre_mean_rr,0,0.00
age,0,0.00
t2m_pre_hf,0,0.00
t2m_pre_lf,0,0.00
t2m_pre_vlf,0,0.00
bp_systolic,0,0.00


In [59]:
# ========= Height consistency check =========
HEIGHT_TOL = 0.01  # 1 cm

df["height_m_from_cm"] = df["height_cm"] / 100.0
df["height_diff_m"] = (df["height_m"] - df["height_m_from_cm"]).abs()

mask_both_height_available = df["height_cm"].notna() & df["height_m"].notna()
df["height_inconsistent"] = False
df.loc[mask_both_height_available, "height_inconsistent"] = (
    df.loc[mask_both_height_available, "height_diff_m"] > HEIGHT_TOL
)

height_inconsistencies = df.loc[
    df["height_inconsistent"],
    ["height_cm", "height_m", "height_m_from_cm", "height_diff_m", "weight_kg", "imc"]
].copy()

height_inconsistencies.to_csv(HEIGHT_INCONSISTENCIES_CSV, index=False)

add_step(
    step_name="check_height_consistency",
    description="Checked consistency between height_cm and height_m.",
    details={
        "tolerance_m": HEIGHT_TOL,
        "n_inconsistent": int(df["height_inconsistent"].sum()),
        "output_file": str(HEIGHT_INCONSISTENCIES_CSV)
    }
)

print("Rows with inconsistent height:", int(df["height_inconsistent"].sum()))
height_inconsistencies.head(20)

Rows with inconsistent height: 9


,height_cm,height_m,height_m_from_cm,height_diff_m,weight_kg,imc
444,148,1.51,1.48,0.03,67.0,29.384676
445,175,1.51,1.75,0.24,65.2,28.595237
446,145,1.51,1.45,0.06,69.8,30.612692
447,148,1.51,1.48,0.03,68.2,29.910969
448,150,1.51,1.50,0.01,68.2,29.910969
449,155,1.51,1.55,0.04,74.1,32.498575
450,149,1.51,1.49,0.02,64.9,28.463664
451,149,1.51,1.49,0.02,59.7,26.183062
452,155,1.51,1.55,0.04,78.1,34.252884


In [60]:
# ========= Build final cleaned height =========
df["height_m_final"] = np.nan
df["height_cm_final"] = np.nan

# Rule 1: if height_cm is available, use it as the primary reference
mask_height_cm_available = df["height_cm"].notna()
df.loc[mask_height_cm_available, "height_cm_final"] = df.loc[mask_height_cm_available, "height_cm"]
df.loc[mask_height_cm_available, "height_m_final"] = df.loc[mask_height_cm_available, "height_cm"] / 100.0

# Rule 2: if height_cm is missing but height_m is available, recover cm from m
mask_only_height_m = df["height_cm"].isna() & df["height_m"].notna()
df.loc[mask_only_height_m, "height_m_final"] = df.loc[mask_only_height_m, "height_m"]
df.loc[mask_only_height_m, "height_cm_final"] = df.loc[mask_only_height_m, "height_m"] * 100.0

add_step(
    step_name="build_final_height_variables",
    description="Constructed final height_m and height_cm variables prioritizing height_cm when available.",
    details={
        "rule": "Use height_cm as primary source; if unavailable, recover from height_m."
    }
)

df[[
    "height_cm", "height_m", "height_cm_final", "height_m_final", "height_inconsistent"
]].head(20)

,height_cm,height_m,height_cm_final,height_m_final,height_inconsistent
0,160,1.60,160.0,1.60,False
1,149,1.49,149.0,1.49,False
2,170,1.70,170.0,1.70,False
3,160,1.60,160.0,1.60,False
4,151,1.51,151.0,1.51,False
5,149,1.49,149.0,1.49,False
6,172,1.72,172.0,1.72,False
7,154,1.54,154.0,1.54,False
8,150,1.50,150.0,1.50,False
9,164,1.64,164.0,1.64,False


In [61]:
# ========= Recompute BMI using final cleaned height =========
df["imc_recomputed_final"] = df["weight_kg"] / (df["height_m_final"] ** 2)
df["imc_diff_final"] = (df["imc"] - df["imc_recomputed_final"]).abs()

BMI_TOL = 0.5
mask_bmi_comparable = df["imc"].notna() & df["imc_recomputed_final"].notna()

df["imc_inconsistent"] = False
df.loc[mask_bmi_comparable, "imc_inconsistent"] = (
    df.loc[mask_bmi_comparable, "imc_diff_final"] > BMI_TOL
)

bmi_inconsistencies = df.loc[
    df["imc_inconsistent"],
    [
        "weight_kg", "height_cm", "height_m", "height_cm_final", "height_m_final",
        "imc", "imc_recomputed_final", "imc_diff_final"
    ]
].copy()

bmi_inconsistencies.to_csv(BMI_INCONSISTENCIES_CSV, index=False)

add_step(
    step_name="recompute_and_check_bmi",
    description="Recomputed BMI using cleaned height and checked consistency against recorded BMI.",
    details={
        "tolerance_bmi_units": BMI_TOL,
        "n_inconsistent": int(df["imc_inconsistent"].sum()),
        "output_file": str(BMI_INCONSISTENCIES_CSV)
    }
)

print("Rows with inconsistent BMI:", int(df["imc_inconsistent"].sum()))
bmi_inconsistencies.head(20)

Rows with inconsistent BMI: 8


,weight_kg,height_cm,height_m,height_cm_final,height_m_final,imc,imc_recomputed_final,imc_diff_final
444,67.0,148,1.51,148.0,1.48,29.384676,30.588020,1.203344
445,65.2,175,1.51,175.0,1.75,28.595237,21.289796,7.305441
446,69.8,145,1.51,145.0,1.45,30.612692,33.198573,2.585881
447,68.2,148,1.51,148.0,1.48,29.910969,31.135866,1.224897
449,74.1,155,1.51,155.0,1.55,32.498575,30.842872,1.655703
450,64.9,149,1.51,149.0,1.49,28.463664,29.232917,0.769254
451,59.7,149,1.51,149.0,1.49,26.183062,26.890681,0.707618
452,78.1,155,1.51,155.0,1.55,34.252884,32.507804,1.745079


In [62]:
# ========= Define final BMI =========
df["imc_final"] = df["imc_recomputed_final"]

add_step(
    step_name="define_final_bmi",
    description="Defined final BMI variable using BMI recomputed from cleaned height.",
    details={
        "final_column": "imc_final"
    }
)

df[["imc", "imc_recomputed_final", "imc_final", "imc_diff_final"]].describe()

,imc,imc_recomputed_final,imc_final,imc_diff_final
count,530.000000,530.000000,530.000000,530.000000
mean,30.713564,30.706366,30.706366,0.033203
std,5.532868,5.544217,5.544217,0.362358
min,18.359375,18.359375,18.359375,0.000000
25%,26.666667,26.666667,26.666667,0.000000
50%,29.733333,29.742700,29.742700,0.000000
75%,33.667215,33.627220,33.627220,0.000000
max,57.074911,57.074911,57.074911,7.305441


In [63]:
# ========= Recompute derived blood pressure variables =========
df["bp_pp_recomputed"] = df["bp_systolic"] - df["bp_diastolic"]
df["bp_pam_recomputed"] = df["bp_diastolic"] + (df["bp_systolic"] - df["bp_diastolic"]) / 3

df["bp_pp_diff"] = (df["bp_pp"] - df["bp_pp_recomputed"]).abs()
df["bp_pam_diff"] = (df["bp_pam"] - df["bp_pam_recomputed"]).abs()

add_step(
    step_name="recompute_bp_variables",
    description="Recomputed pulse pressure and mean arterial pressure for consistency checks.",
    details={
        "new_columns": [
            "bp_pp_recomputed", "bp_pam_recomputed", "bp_pp_diff", "bp_pam_diff"
        ]
    }
)

df[["bp_pp", "bp_pp_recomputed", "bp_pp_diff", "bp_pam", "bp_pam_recomputed", "bp_pam_diff"]].describe()

,bp_pp,bp_pp_recomputed,bp_pp_diff,bp_pam,bp_pam_recomputed,bp_pam_diff
count,530.000000,530.000000,530.0,530.000000,530.000000,5.300000e+02
mean,52.779245,52.779245,0.0,95.070440,95.070440,2.681293e-17
std,14.142080,14.142080,0.0,11.141434,11.141434,6.172801e-16
min,16.000000,16.000000,0.0,66.000000,66.000000,0.000000e+00
25%,43.000000,43.000000,0.0,87.666667,87.666667,0.000000e+00
50%,51.000000,51.000000,0.0,94.333333,94.333333,0.000000e+00
75%,61.000000,61.000000,0.0,102.000000,102.000000,0.000000e+00
max,101.000000,101.000000,0.0,135.666667,135.666667,1.421085e-14


In [64]:
# ========= HR-RR consistency check =========
# HR approximately equals 60000 / RR(ms)

df["mean_hr_from_rr"] = 60000 / df["t2m_pre_mean_rr"]
df["mean_hr_rr_diff"] = (df["t2m_pre_mean_hr"] - df["mean_hr_from_rr"]).abs()

HR_RR_TOL = 2.0  # bpm tolerance

mask_hr_rr_comparable = df["t2m_pre_mean_hr"].notna() & df["t2m_pre_mean_rr"].notna()
df["hr_rr_inconsistent"] = False
df.loc[mask_hr_rr_comparable, "hr_rr_inconsistent"] = (
    df.loc[mask_hr_rr_comparable, "mean_hr_rr_diff"] > HR_RR_TOL
)

hr_rr_inconsistencies = df.loc[
    df["hr_rr_inconsistent"],
    ["t2m_pre_mean_rr", "t2m_pre_mean_hr", "mean_hr_from_rr", "mean_hr_rr_diff"]
].copy()

hr_rr_inconsistencies.to_csv(HR_RR_INCONSISTENCIES_CSV, index=False)

add_step(
    step_name="check_hr_rr_consistency",
    description="Checked consistency between mean heart rate and mean RR interval.",
    details={
        "tolerance_bpm": HR_RR_TOL,
        "n_inconsistent": int(df["hr_rr_inconsistent"].sum()),
        "output_file": str(HR_RR_INCONSISTENCIES_CSV)
    }
)

print("Rows with HR-RR inconsistency:", int(df["hr_rr_inconsistent"].sum()))
hr_rr_inconsistencies.head(20)

Rows with HR-RR inconsistency: 15


,t2m_pre_mean_rr,t2m_pre_mean_hr,mean_hr_from_rr,mean_hr_rr_diff
118,100.0,60.0,600.000000,540.000000
154,1036.0,104.0,57.915058,46.084942
181,1014.0,56.0,59.171598,3.171598
182,68.0,88.0,882.352941,794.352941
185,106.0,57.0,566.037736,509.037736
190,100.0,60.0,600.000000,540.000000
191,600.0,10.0,100.000000,90.000000
200,106.0,57.0,566.037736,509.037736
212,105.0,57.0,571.428571,514.428571
260,85.8,70.0,699.300699,629.300699


In [65]:
# ========= Physiological range checks =========
range_rules = {
    "age": (18, 100),
    "weight_kg": (30, 250),
    "height_cm_final": (120, 230),
    "height_m_final": (1.2, 2.3),
    "imc_final": (10, 80),
    "bp_systolic": (70, 250),
    "bp_diastolic": (40, 150),
    "bp_pam": (40, 180),
    "bp_pp": (10, 150),
    "t2m_pre_mean_hr": (30, 220),
    "t2m_pre_mean_rr": (250, 2000),
    "t2m_pre_sdnn": (0, 500),
    "t2m_pre_rmssd": (0, 500),
    "t2m_pre_hf": (0, 100000),
    "t2m_pre_lf": (0, 100000),
    "t2m_pre_vlf": (0, 100000),
}

range_flags = pd.DataFrame(index=df.index)

for col, (lower, upper) in range_rules.items():
    if col in df.columns:
        range_flags[f"{col}_out_of_range"] = (
            df[col].notna() & ((df[col] < lower) | (df[col] > upper))
        )

range_flags["n_flags"] = range_flags.sum(axis=1)

range_flag_rows = pd.concat([df.copy(), range_flags], axis=1)
range_flag_rows = range_flag_rows.loc[range_flags["n_flags"] > 0].copy()

range_flag_rows.to_csv(RANGE_FLAGS_CSV, index=False)

range_summary = {
    col: int(range_flags[col].sum())
    for col in range_flags.columns
    if col != "n_flags"
}

add_step(
    step_name="physiological_range_checks",
    description="Flagged potentially implausible values based on broad physiological ranges.",
    details={
        "range_rules": {k: {"min": v[0], "max": v[1]} for k, v in range_rules.items()},
        "n_rows_with_any_flag": int((range_flags["n_flags"] > 0).sum()),
        "flag_counts": range_summary,
        "output_file": str(RANGE_FLAGS_CSV)
    }
)

print("Rows with at least one physiological range flag:", int((range_flags["n_flags"] > 0).sum()))
range_flag_rows.head(20)

Rows with at least one physiological range flag: 14


,sex,age,t2m_pre_mean_rr,t2m_pre_mean_hr,t2m_pre_sdnn,t2m_pre_rmssd,t2m_pre_hf,t2m_pre_lf,t2m_pre_vlf,bp_systolic,...,bp_pam_out_of_range,bp_pp_out_of_range,t2m_pre_mean_hr_out_of_range,t2m_pre_mean_rr_out_of_range,t2m_pre_sdnn_out_of_range,t2m_pre_rmssd_out_of_range,t2m_pre_hf_out_of_range,t2m_pre_lf_out_of_range,t2m_pre_vlf_out_of_range,n_flags
118,1,79,100.0,60.0,181.0,25.8,229,49,36,138,...,False,False,False,True,False,False,False,False,False,1
122,2,71,833.0,72.0,626.0,59.4,5512,614,275,147,...,False,False,False,False,True,False,False,False,False,1
182,1,67,68.0,88.0,71.0,4.5,10,37,7,146,...,False,False,False,True,False,False,False,False,False,1
185,1,87,106.0,57.0,241.0,25.7,1585,166,8,154,...,False,False,False,True,False,False,False,False,False,1
190,1,66,100.0,60.0,145.0,17.1,258,62,20,114,...,False,False,False,True,False,False,False,False,False,1
191,1,73,600.0,10.0,239.0,10.4,206,334,66,136,...,False,False,True,False,False,False,False,False,False,1
200,2,79,106.0,57.0,121.0,9.2,850,1630,40,116,...,False,False,False,True,False,False,False,False,False,1
202,2,76,1003.0,60.0,610.0,71.6,4853,719,97,121,...,False,False,False,False,True,False,False,False,False,1
212,1,67,105.0,57.0,284.0,19.8,894,781,180,124,...,False,False,False,True,False,False,False,False,False,1
240,2,71,962.0,62.0,525.0,71.6,1998,336,20,126,...,False,False,False,False,True,False,False,False,False,1


In [66]:
# ========= Row-level integrity summary =========
integrity_cols = [
    "height_inconsistent",
    "imc_inconsistent",
    "hr_rr_inconsistent"
]

existing_integrity_cols = [col for col in integrity_cols if col in df.columns]

df["n_integrity_flags"] = df[existing_integrity_cols].sum(axis=1)

if "n_flags" in range_flags.columns:
    df["n_range_flags"] = range_flags["n_flags"]
else:
    df["n_range_flags"] = 0

df["n_total_qc_flags"] = df["n_integrity_flags"] + df["n_range_flags"]

add_step(
    step_name="row_level_qc_summary",
    description="Created row-level quality control summary flags.",
    details={
        "integrity_flag_columns": existing_integrity_cols,
        "includes_range_flags": True
    }
)

df[["n_integrity_flags", "n_range_flags", "n_total_qc_flags"]].describe()

,n_integrity_flags,n_range_flags,n_total_qc_flags
count,530.000000,530.000000,530.000000
mean,0.060377,0.026415,0.086792
std,0.295102,0.160518,0.388900
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,2.000000,1.000000,2.000000


In [67]:
# ========= Duplicate and empty row checks =========
duplicate_rows = int(df.duplicated().sum())
all_missing_rows = int(df.isna().all(axis=1).sum())

add_step(
    step_name="row_integrity_checks",
    description="Checked duplicated rows and fully missing rows.",
    details={
        "duplicate_rows": duplicate_rows,
        "fully_missing_rows": all_missing_rows
    }
)

print("Duplicate rows:", duplicate_rows)
print("Fully missing rows:", all_missing_rows)

Duplicate rows: 0
Fully missing rows: 0


In [68]:
# ========= Final analysis-ready variable selection =========
analysis_cols = [
    "record_id" if "record_id" in df.columns else None,
    "sex",
    "age",
    "weight_kg",
    "height_cm_final",
    "height_m_final",
    "imc_final",
    "bp_systolic",
    "bp_diastolic",
    "bp_pam",
    "bp_pp",
    "bp_pp_recomputed",
    "bp_pam_recomputed",
    "t2m_pre_mean_rr",
    "t2m_pre_mean_hr",
    "t2m_pre_sdnn",
    "t2m_pre_rmssd",
    "t2m_pre_hf",
    "t2m_pre_lf",
    "t2m_pre_vlf",
    "mean_hr_from_rr",
    "height_inconsistent",
    "imc_inconsistent",
    "hr_rr_inconsistent",
    "n_integrity_flags",
    "n_range_flags",
    "n_total_qc_flags",
]

analysis_cols = [col for col in analysis_cols if col is not None and col in df.columns]

df_final = df[analysis_cols].copy()

add_step(
    step_name="build_analysis_dataset",
    description="Built final analysis-ready dataset with cleaned core variables and QC flags.",
    details={
        "analysis_columns": analysis_cols
    }
)

print("Final analysis dataset shape:", df_final.shape)
df_final.head()

Final analysis dataset shape: (530, 26)


,sex,age,weight_kg,height_cm_final,height_m_final,imc_final,bp_systolic,bp_diastolic,bp_pam,bp_pp,...,t2m_pre_hf,t2m_pre_lf,t2m_pre_vlf,mean_hr_from_rr,height_inconsistent,imc_inconsistent,hr_rr_inconsistent,n_integrity_flags,n_range_flags,n_total_qc_flags
0,1,75,70.0,160.0,1.60,27.343750,108,70,82.666667,38,...,222,713,172,80.213904,False,False,False,0,0,0
1,1,66,52.0,149.0,1.49,23.422368,122,72,88.666667,50,...,87,226,29,62.959077,False,False,False,0,0,0
2,2,77,81.0,170.0,1.70,28.027682,125,85,98.333333,40,...,26,137,9,68.415051,False,False,False,0,0,0
3,1,77,85.0,160.0,1.60,33.203125,126,72,90.000000,54,...,118,130,14,73.260073,False,False,False,0,0,0
4,1,73,69.0,151.0,1.51,30.261831,130,80,96.666667,50,...,66,66,18,70.921986,False,False,False,0,0,0


In [69]:
# ========= Final diagnostic =========
final_diagnostic = pd.DataFrame({
    "dtype_final": df_final.dtypes.astype(str),
    "missing_n": df_final.isna().sum(),
    "missing_pct": (df_final.isna().mean() * 100).round(2),
    "n_unique": df_final.nunique(dropna=True)
}).sort_values(["missing_n", "dtype_final"], ascending=[False, True])

final_diagnostic.to_csv(FINAL_DIAGNOSTIC_CSV)
final_diagnostic

,dtype_final,missing_n,missing_pct,n_unique
t2m_pre_sdnn,float64,11,2.08,268
t2m_pre_rmssd,float64,8,1.51,215
t2m_pre_mean_hr,float64,1,0.19,55
height_inconsistent,bool,0,0.00,2
imc_inconsistent,bool,0,0.00,2
hr_rr_inconsistent,bool,0,0.00,2
weight_kg,float64,0,0.00,210
height_cm_final,float64,0,0.00,40
height_m_final,float64,0,0.00,40
imc_final,float64,0,0.00,411


In [70]:
# ========= Save cleaned dataset =========
df_final.to_csv(CLEAN_CSV, index=False)
df_final.to_excel(CLEAN_XLSX, index=False)

add_step(
    step_name="save_clean_dataset",
    description="Saved cleaned analysis-ready dataset in CSV and XLSX formats.",
    details={
        "csv_path": str(CLEAN_CSV),
        "xlsx_path": str(CLEAN_XLSX)
    }
)

print("Saved cleaned dataset:")
print(CLEAN_CSV)
print(CLEAN_XLSX)

Saved cleaned dataset:
../../results/processed_hrv/basal_v2_clean.csv
../../results/processed_hrv/basal_v2_clean.xlsx


In [71]:
# ========= Finalize and save processing log =========
processing_log["final_shape"] = {
    "n_rows": int(df_final.shape[0]),
    "n_cols": int(df_final.shape[1]),
}

processing_log["output_files"] = {
    "clean_csv": str(CLEAN_CSV),
    "clean_xlsx": str(CLEAN_XLSX),
    "processing_json": str(PROCESSING_JSON),
    "initial_diagnostic_csv": str(INITIAL_DIAGNOSTIC_CSV),
    "final_diagnostic_csv": str(FINAL_DIAGNOSTIC_CSV),
    "height_inconsistencies_csv": str(HEIGHT_INCONSISTENCIES_CSV),
    "bmi_inconsistencies_csv": str(BMI_INCONSISTENCIES_CSV),
    "hr_rr_inconsistencies_csv": str(HR_RR_INCONSISTENCIES_CSV),
    "range_flags_csv": str(RANGE_FLAGS_CSV),
}

with open(PROCESSING_JSON, "w", encoding="utf-8") as f:
    json.dump(processing_log, f, indent=4, ensure_ascii=False)

print("Processing log saved to:", PROCESSING_JSON)

Processing log saved to: ../../results/processed_hrv/processing_log.json


In [72]:
# ========= Quick inspection of processing log =========
with open(PROCESSING_JSON, "r", encoding="utf-8") as f:
    log_preview = json.load(f)

log_preview

{'dataset_name': 'Basal v2',
 'timestamp': '2026-03-12T17:16:25.381937',
 'input_file': '../../raw_data/Basal v2.xlsx',
 'initial_shape': {'n_rows': 530, 'n_cols': 20},
 'steps': [{'step': 'drop_fully_empty_columns',
   'description': 'Removed columns containing only missing values.',
   'details': {'dropped_columns': ['Unnamed: 0',
     'Unnamed: 1',
     'redcap_repeat_instrument'],
    'n_dropped': 3}},
  {'step': 'rename_columns',
   'description': 'Renamed columns for consistency and easier downstream processing.',
   'details': {'rename_map': {'weight kg': 'weight_kg',
     'height cm': 'height_cm',
     'height mt': 'height_m'}}},
  {'step': 'clean_object_numeric_strings',
   'description': 'Applied generic malformed numeric-string cleaning to object columns.',
   'details': {'object_columns_cleaned': ['t2m_pre_rmssd']}},
  {'step': 'force_numeric_conversion',
   'description': 'Forced selected variables to numeric type using coercion.',
   'details': {'numeric_columns': ['sex',

In [73]:
# ========= QC summary =========
qc_summary = {
    "n_rows_raw": int(df_raw.shape[0]),
    "n_cols_raw": int(df_raw.shape[1]),
    "n_rows_final": int(df_final.shape[0]),
    "n_cols_final": int(df_final.shape[1]),
    "n_height_inconsistent": int(df["height_inconsistent"].sum()) if "height_inconsistent" in df.columns else 0,
    "n_bmi_inconsistent": int(df["imc_inconsistent"].sum()) if "imc_inconsistent" in df.columns else 0,
    "n_hr_rr_inconsistent": int(df["hr_rr_inconsistent"].sum()) if "hr_rr_inconsistent" in df.columns else 0,
    "n_rows_with_any_qc_flag": int((df["n_total_qc_flags"] > 0).sum()) if "n_total_qc_flags" in df.columns else 0,
}

pd.Series(qc_summary, name="value")

n_rows_raw                 530
n_cols_raw                  20
n_rows_final               530
n_cols_final                26
n_height_inconsistent        9
n_bmi_inconsistent           8
n_hr_rr_inconsistent        15
n_rows_with_any_qc_flag     27
Name: value, dtype: int64